# Objetivos

- [Gráfico 1](#gr1): taxa de sucesso na realocação. Usuários afetados que foram realocados devidamente dividido pelo total de usuários afetados


- [Gráfico 2](#gr2): Taxa de utilização de Espaço da Rede: que é quão bem a abordagem está aproveitando o espaço de armazenamento disponível após uma falha na rede. Uma taxa alta indica uma boa utilização dos recursos 

- [Gráfico 3](#gr3): -Taxa de perda de benefícios: O quão o usuário foi prejudicado nessa realocação em comparação com a alocação antiga dele


- [Gráfico 4](#gr4): - tempo de realocação

### Configuração inicial

Importando bibliotecas necessárias

In [15]:
import matplotlib.pyplot as plt
from typing import List
import glob
import os
import numpy as np
import pandas as pd 
# import seaborn as sns
#import plotly.graph_objects as go
import numpy as np
#import plotly.io as pio
#import plotly.express as px


import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter('ignore')

### Carregamento dos dados

In [16]:
results_res_directories = glob.glob('../../results/results_resilient*/*') # Caso mude a estrutura dos diretórios, esse caminho deve ser alterado
results_res_directories

['../../results\\results_resilient\\goku_s_50_p_50_sfc_on',
 '../../results\\results_resilient\\old_s_50_p_50_sfc_on',
 '../../results\\results_resilient\\vegeta_s_50_p_50_sfc_on']

In [17]:
for alg_dir in results_res_directories:
    simulacoes_okays = 0
    simu_exec_name = alg_dir.split('/')[-1].split('_')
    print(simu_exec_name)
    alg_name = simu_exec_name[1].split(f'\\')[1]
    print(alg_name)

['results\\results', 'resilient\\goku', 's', '50', 'p', '50', 'sfc', 'on']
goku
['results\\results', 'resilient\\old', 's', '50', 'p', '50', 'sfc', 'on']
old
['results\\results', 'resilient\\vegeta', 's', '50', 'p', '50', 'sfc', 'on']
vegeta


In [18]:
aggregated_simulation_data = pd.DataFrame() 
algos = {}
def collect_data_from_alg_directory(results_res_directories):
    data_nla = []
    log_simu = {}  # Essa variável serve apenas para verificar as simulações

    for alg_dir in results_res_directories:
        simulacoes_okays = 0
        simu_exec_name = alg_dir.split('/')[-1].split('_')
        
        # if simu_exec_name [2] == 'backup':
        #     alg_name = 'goku_backup'
        # else:
        alg_name = simu_exec_name[1].split(f'\\')[1]
        # reliability = simu_exec_name[-1]
        if alg_name == 'g':
            alg_name = 'Greedy'
        if alg_name == 'ga':
            alg_name =  'GA'
        if alg_name == 'msf':
            alg_name = 'MSF'
        if alg_name == 'goku':
            alg_name = 'OSCIM'
        if alg_name == 'vegeta':
            alg_name = 'Resilient-OSCIM'
        if alg_name == 'musfico':
            alg_name = 'MuSFiCO'
        if alg_name == 'old':
            alg_name = 'All Backup'
        files = os.listdir(alg_dir)
        #print(f"Simulação: {alg_name}, {reliability}") 
        #print("Quantidade de csv: ", len(files))
        
        if alg_name not in log_simu:
            log_simu[alg_name] = {'files': len(files), 'simulacoes_sucesso': 0, 'Nulos': 0}
        else:
            log_simu[alg_name]['files'] += len(files)

        for file in files:
            data_path = os.path.join(alg_dir, file)
            try:
                simulation_df = pd.read_csv(data_path)
            except Exception as e:
                print(f"Erro ao ler o arquivo {file}: {e}")
                continue

            if not simulation_df.empty: 
                # Calculations
                mean_data_frame = pd.DataFrame()
                recover_success_rate = simulation_df["recover_success"].mean()
                backup_success_rate = simulation_df["backup_success"].mean()
                latency_diff_mean = simulation_df["latency_diff"].dropna().mean()
                time_to_recover_mean = simulation_df["time_to_recover"].dropna().mean()

                # Cálculo do percentual de melhoria da latência
                improvement_percentage = ((5 - latency_diff_mean) / 10) * 100

                mean_data_frame["recover_success"] = [recover_success_rate]
                mean_data_frame["backup_success"] = [backup_success_rate]
                mean_data_frame["latency_diff"] = [latency_diff_mean]
                mean_data_frame["latency_improvement"] = [improvement_percentage]
                mean_data_frame["time_to_recover"] = [time_to_recover_mean]

                # Seleciona as métricas relevantes
                mean_data_frame['algorithm'] = alg_name
                mean_data_frame['simulation'] = log_simu[alg_name]['simulacoes_sucesso']
                print(mean_data_frame)
                # print(mean_data_frame)
                data_nla.append(mean_data_frame)

                # Incrementa a contagem de simulações de sucesso e soma os valores nulos
                log_simu[alg_name]['simulacoes_sucesso'] += 1
                log_simu[alg_name]['Nulos'] += sum(mean_data_frame.isnull().sum())
    if data_nla:
        data_nla_f = pd.concat(data_nla)
    else:
        data_nla_f = pd.DataFrame()

    print("\nResumo das Simulações:")
    for alg, info in log_simu.items():
        print(f"Algoritmo: {alg}")
        print(f"  Quantidade de arquivos: {info['files']}")
        print(f"  Simulações de sucesso: {info['simulacoes_sucesso']}")
        print(f"  Dados nulos: {info['Nulos']}")
        print()

    return data_nla_f

In [19]:
aggregated_simulation_data = collect_data_from_alg_directory(results_res_directories)

   recover_success  backup_success  latency_diff  latency_improvement  \
0              0.8             0.0        -0.625                56.25   

   time_to_recover algorithm  simulation  
0        23.746594     OSCIM           0  
   recover_success  backup_success  latency_diff  latency_improvement  \
0            0.875             0.0      2.142857            28.571429   

   time_to_recover algorithm  simulation  
0        17.952387     OSCIM           1  
   recover_success  backup_success  latency_diff  latency_improvement  \
0             0.25             0.0          1.25                 37.5   

   time_to_recover algorithm  simulation  
0        17.313726     OSCIM           2  
   recover_success  backup_success  latency_diff  latency_improvement  \
0         0.923077             0.0      0.916667            40.833333   

   time_to_recover algorithm  simulation  
0        16.627677     OSCIM           3  
   recover_success  backup_success  latency_diff  latency_improvemen

In [20]:
aggregated_simulation_data

,recover_success,backup_success,latency_diff,latency_improvement,time_to_recover,algorithm,simulation
0,0.800000,0.000000,-0.625000,56.250000,23.746594,OSCIM,0
0,0.875000,0.000000,2.142857,28.571429,17.952387,OSCIM,1
0,0.250000,0.000000,1.250000,37.500000,17.313726,OSCIM,2
0,0.923077,0.000000,0.916667,40.833333,16.627677,OSCIM,3
0,1.000000,0.000000,-0.111111,51.111111,29.377863,OSCIM,4
0,0.727273,0.000000,-1.750000,67.500000,13.440515,OSCIM,5
0,0.700000,0.000000,2.142857,28.571429,20.696123,OSCIM,6
0,1.000000,0.000000,-1.222222,62.222222,19.228258,OSCIM,7
0,1.000000,0.000000,-0.428571,54.285714,29.653628,OSCIM,8
0,1.000000,0.000000,0.000000,50.000000,23.384817,OSCIM,9


# Gráfico 1 <a id="gr1"></a>

taxa de sucesso na realocação. Usuários afetados que foram realocados devidamente dividido pelo total de usuários afetados


In [22]:
! pip install kaleido

In [23]:
import plotly.express as px
import plotly.graph_objects as go


df = aggregated_simulation_data


# Lista de métricas
metrics = ["recover_success", "backup_success", "latency_diff","latency_improvement", "time_to_recover"]


# Criação de um gráfico para cada métrica
for metric in metrics:
    fig = go.Figure()
    for algo in df['algorithm'].unique():
        algo_data = df[df['algorithm'] == algo]
        fig.add_trace(go.Box(
            y=algo_data[metric],
            name=algo,
            boxmean=True,
            marker=dict(size=6),
            showlegend=True
        ))
    
    fig.update_layout(
        yaxis_title=metric,
        xaxis_title="Algorithm",
        boxmode='group',
        legend=dict(
            orientation="h",  # Legenda na horizontal
            yanchor="bottom",
            y=1.05,  # Coloca a legenda logo acima do gráfico
            xanchor="center",
            x=0.5,
            font=dict(size=14)  # Aumenta o tamanho da fonte da legenda
        ),
        autosize=True,  # Permite o ajuste automático do tamanho
        margin=dict(l=50, r=50, t=10, b=50),  # Ajustes nas margens para mais flexibilidade
        height=400,  # Ajuste flexível no tamanho do plot
        width=600,  # Ajuste flexível no tamanho do plot
        font=dict(
            size=16  # Aumenta o tamanho geral da fonte
        )
    )

    # Ajusta o tamanho da fonte dos eixos
    fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=14))
    fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=14))
    # Nome do arquivo baseado na métrica
    file_name = f"{metric}.pdf"
    fig.write_image(file_name, format="pdf")
    fig.show()
